# 04 — Validate the Cross-Encoder Relevance Model

This notebook downloads `cross-encoder/ms-marco-MiniLM-L6-v2` from HuggingFace,
validates its behaviour against known query-snippet pairs, then saves the weights
to `models/relevance/` so the model service loads from disk — not from HuggingFace
at runtime.

We ship the model we tested. Not a model we assume is the same.

**What this notebook validates:**
- Relevant snippets score higher than irrelevant ones
- Ranking order is correct across a multi-snippet query
- Model generalises across all 6 intent classes
- Scores interact correctly with the sufficiency formula and per-class thresholds
- Inference latency is under 5ms per pair on CPU

**Input:**  HuggingFace (download only, once)  
**Output:** `models/relevance/` — pinned weights ready to copy to opensearch


## 1. Imports and configuration


In [1]:
import time
import numpy as np
from pathlib import Path
from sentence_transformers import CrossEncoder
import warnings
warnings.filterwarnings('ignore')

MODEL_NAME  = 'cross-encoder/ms-marco-MiniLM-L6-v2'
MODEL_DIR   = Path('../models/relevance')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

INTENT_CLASSES = ['news', 'factual', 'code', 'research', 'commercial', 'general']

# sufficiency thresholds from CLASSIFIERS.md
SUFFICIENCY_THRESHOLDS = {
    'factual': 0.70,
    'code': 0.65,
    'general': 0.65,
    'news': 0.60,
    'commercial': 0.60,
    'research': 0.45,
}

print(f'Model : {MODEL_NAME}')
print(f'Output dir : {MODEL_DIR.resolve()}')


Model          : cross-encoder/ms-marco-MiniLM-L6-v2
Output dir     : /home/festus/ml/opensearch-models/models/relevance


## 2. Download and load the model

Downloads from HuggingFace once. All subsequent validation runs against
this in-memory instance, which we then save to disk at the end.


In [2]:
t0 = time.time()
model = CrossEncoder(MODEL_NAME)
elapsed = time.time() - t0

print(f'Model loaded in {elapsed:.2f}s')
print(f'Max input length : {model.max_seq_length}')


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded in 30.68s
Max input length : 512


## 3. Sanity check — relevant vs irrelevant

The model outputs raw logits, not probabilities. The absolute value is not
meaningful in isolation — what matters is that relevant snippets score
significantly higher than irrelevant ones for the same query.

These are the exact examples from CLASSIFIERS.md.


In [3]:
sanity_pairs = [
    (
        'what year was Redis released',
        'Redis was first released in 2009 by Salvatore Sanfilippo',
        'relevant',
    ),
    (
        'what year was Redis released',
        'Redis supports two persistence modes',
        'irrelevant',
    ),
    (
        'explain Redis persistence tradeoffs RDB vs AOF',
        'RDB snapshots the dataset at intervals while AOF logs every write operation, '
        'each with distinct durability and performance tradeoffs',
        'relevant',
    ),
    (
        'explain Redis persistence tradeoffs RDB vs AOF',
        'Redis was first released in 2009 by Salvatore Sanfilippo',
        'irrelevant',
    ),
]

pairs = [(q, s) for q, s, _ in sanity_pairs]
labels = [l for _, _, l in sanity_pairs]
scores = model.predict(pairs)

print('Sanity check — relevant vs irrelevant:')
print()
print(f'{"query":<45}  {"label":<10}  {"score":>8}')
print('-' * 70)
for (q, s, label), score in zip(sanity_pairs, scores):
    print(f'{q:<45}  {label:<10}  {score:>8.4f}')

print()
# verify relevant always beats irrelevant for the same query
rel_1, irrel_1, rel_2, irrel_2 = scores
assert rel_1 > irrel_1, f'FAIL: relevant score {rel_1:.4f} did not beat irrelevant {irrel_1:.4f} for query 1'
assert rel_2 > irrel_2, f'FAIL: relevant score {rel_2:.4f} did not beat irrelevant {irrel_2:.4f} for query 2'
print('PASS  relevant scores higher than irrelevant for both queries')


Sanity check — relevant vs irrelevant:

query                                          label          score
----------------------------------------------------------------------
what year was Redis released                   relevant      9.4490
what year was Redis released                   irrelevant   -4.1535
explain Redis persistence tradeoffs RDB vs AOF  relevant      1.9160
explain Redis persistence tradeoffs RDB vs AOF  irrelevant   -8.4899

PASS  relevant scores higher than irrelevant for both queries


## 4. Per-intent-class validation

One clearly relevant and one clearly irrelevant snippet per intent class.
Confirms the model generalises across the query types the system will receive.


In [15]:
intent_pairs = {
    'news': [
        (
            'premier league scores this weekend',
            'Manchester City beat Arsenal 2-1 on Saturday with goals from Haaland '
            'and De Bruyne to go top of the Premier League table',
            'relevant',
        ),
        (
            'premier league scores this weekend',
            'The Premier League was founded in 1992 and currently has 20 clubs',
            'irrelevant',
        ),
    ],
    'factual': [
        (
            'capital of france',
            'Paris is the capital and most populous city of France',
            'relevant',
        ),
        (
            'capital of france',
            'France is a country in Western Europe known for its wine and cuisine',
            'irrelevant',
        ),
    ],
    'code': [
        (
            'how to handle goroutine panic in go',
            'Use a deferred recover call inside the goroutine to catch panics. '
            'recover returns the value passed to panic and stops the propagation',
            'relevant',
        ),
        (
            'how to handle goroutine panic in go',
            'Go is a statically typed compiled language designed at Google',
            'irrelevant',
        ),
    ],
    'research': [
        (
            'cap theorem distributed systems',
            'The CAP theorem states that a distributed system can provide at most two '
            'of three guarantees: consistency, availability, and partition tolerance',
            'relevant',
        ),
        (
            'cap theorem distributed systems',
            'Distributed systems are used in many large scale applications today',
            'irrelevant',
        ),
    ],
    'commercial': [
        (
            'best laptop under 1000 dollars for programming',
            'The Lenovo ThinkPad E14 at $899 offers the best value for developers '
            'with its strong keyboard, 16GB RAM, and 12-hour battery life',
            'relevant',
        ),
        (
            'best laptop under 1000 dollars for programming',
            'Laptops have become essential tools for both work and entertainment',
            'irrelevant',
        ),
    ],
    'general': [
    (
        'tips for better sleep',
        'Keep a consistent sleep schedule, avoid screens one hour before bed, '
        'keep your bedroom cool and dark, and limit caffeine after 2pm',
        'relevant',
    ),
    (
        'tips for better sleep',
        'Sleep is a natural recurring state of rest for the mind and body',
        'irrelevant',
    ),
],
}

print('Per-intent-class validation:')
print()
print(f'{"intent":<12}  {"label":<10}  {"score":>8}  result')
print('-' * 55)

all_passed = True
for intent, pairs in intent_pairs.items():
    texts = [(q, s) for q, s, _ in pairs]
    scores = model.predict(texts)
    rel_score   = scores[0]
    irrel_score = scores[1]
    passed = rel_score > irrel_score
    if not passed:
        all_passed = False
    for (q, s, label), score in zip(pairs, scores):
        result = ('PASS' if passed else 'FAIL') if label == 'relevant' else ''
        print(f'{intent:<12}  {label:<10}  {score:>8.4f}  {result}')

print()
if all_passed:
    print('PASS  relevant outscores irrelevant across all 6 intent classes')
else:
    print('FAIL  one or more intent classes scored incorrectly')


Per-intent-class validation:

intent        label          score  result
-------------------------------------------------------
news          relevant      3.7420  PASS
news          irrelevant   -4.2369  
factual       relevant      7.6027  PASS
factual       irrelevant   -1.6193  
code          relevant      6.0532  PASS
code          irrelevant  -10.8020  
research      relevant      9.6312  PASS
research      irrelevant   -3.4170  
commercial    relevant     -1.3414  PASS
commercial    irrelevant   -9.2606  
general       relevant      1.6034  PASS
general       irrelevant   -6.4996  

PASS  relevant outscores irrelevant across all 6 intent classes


## 5. Ranking validation

Given one query and multiple snippets of varying relevance, confirm the model
ranks them in the correct order. This directly simulates how the crawler module
uses the /relevance endpoint — top 3 results scored and ranked.


In [6]:
query = 'how to implement an LRU cache in Go'

snippets = [
    (
        'An LRU cache in Go can be implemented using a doubly linked list combined '
        'with a hash map. The list tracks access order, the map provides O(1) '
        'lookup. On eviction, remove the tail node.',
        'direct implementation answer',
    ),
    (
        'The groupcache library by Google provides a distributed LRU cache '
        'implementation in Go that is used in production at scale',
        'related but not a how-to',
    ),
    (
        'Caching is a technique used to store frequently accessed data in fast '
        'memory to reduce latency',
        'generic topic mention',
    ),
    (
        'Go was designed at Google and released as open source in 2009',
        'off-topic',
    ),
]

pairs = [(query, s) for s, _ in snippets]
scores = model.predict(pairs)

ranked = sorted(zip(scores, snippets), reverse=True)

print(f'Query: {query}')
print()
print(f'{"rank":<6}  {"score":>8}  description')
print('-' * 70)
for rank, (score, (snippet, desc)) in enumerate(ranked, 1):
    print(f'{rank:<6}  {score:>8.4f}  {desc}')

print()
top_desc = ranked[0][1][1]
assert top_desc == 'direct implementation answer', \
    f'FAIL: expected direct implementation answer at rank 1, got: {top_desc}'
print('PASS  direct implementation answer ranked first')


Query: how to implement an LRU cache in Go

rank       score  description
----------------------------------------------------------------------
1         9.1435  direct implementation answer
2         5.6538  related but not a how-to
3       -10.2528  off-topic
4       -11.2879  generic topic mention

PASS  direct implementation answer ranked first


## 6. Sufficiency threshold simulation

Plug relevance scores into the actual sufficiency formula from CLASSIFIERS.md
and verify the crawl decision triggers correctly per intent class.

```
sufficiency = (relevance_score * 0.50)
            + (snippet_density * 0.30)
            + (source_authority * 0.20)
```


In [18]:
def sufficiency_score(relevance, snippet_density, source_authority):
    return (relevance * 0.50) + (snippet_density * 0.30) + (source_authority * 0.20)

def crawl_decision(sufficiency, intent):
    threshold = SUFFICIENCY_THRESHOLDS[intent]
    return 'snippets sufficient — skip crawl' if sufficiency >= threshold else 'crawl required'

# simulate top 3 results per intent class
# using a clearly relevant snippet, a medium snippet, and a weak snippet
# snippet_density and source_authority are fixed representative values
simulation = [
    ('factual',    'what is the capital of france',
                   'Paris is the capital and most populous city of France', 0.75, 0.80),
    ('code',       'how to handle goroutine panic in go',
                   'Use defer recover inside the goroutine to catch and handle panics', 0.70, 0.65),
    ('general',    'tips for better sleep',
                   'Bring water to a boil then gradually add maize flour while stirring', 0.68, 0.60),
    ('news',       'premier league scores this weekend',
                   'The commission announced results will be declared by 5pm after tallying', 0.65, 0.55),
    ('commercial', 'best laptop under 1000 dollars for programming',
                   'The ThinkPad E14 at 899 dollars offers the best value for developers', 0.60, 0.70),
    ('research', 'cap theorem distributed systems',
                   'CAP theorem states a system can guarantee at most two of three properties', 0.55, 0.75),
]

print('Sufficiency threshold simulation:')
print()
print(f'{"intent":<12}  {"rel_score":>9}  {"sufficiency":>11}  {"threshold":>9}  decision')
print('-' * 80)

for intent, query, snippet, density, authority in simulation:
    raw_logit = model.predict([(query, snippet)])[0]
    rel_normalised = float(1 / (1 + np.exp(-raw_logit)))
    suf = sufficiency_score(rel_normalised, density, authority)
    threshold = SUFFICIENCY_THRESHOLDS[intent]
    decision = crawl_decision(suf, intent)
    print(f'{intent:<12}  {rel_normalised:>9.4f}  {suf:>11.4f}  {threshold:>9.2f}  {decision}')


Sufficiency threshold simulation:

intent        rel_score  sufficiency  threshold  decision
--------------------------------------------------------------------------------
factual          0.9994       0.8847       0.70  snippets sufficient — skip crawl
code             0.9990       0.8395       0.65  snippets sufficient — skip crawl
general          0.0000       0.3240       0.65  crawl required
news             0.0000       0.3050       0.60  crawl required
commercial       0.4764       0.5582       0.60  crawl required
research         0.9976       0.8138       0.45  snippets sufficient — skip crawl


## 7. Latency check

Confirm inference stays under 5ms per pair on CPU as the architecture specifies.
Tested on a batch of 50 pairs to get a stable average.


In [14]:
latency_pairs = [
    ('how to handle goroutine panic in go',
     'Use a deferred recover call inside the goroutine to catch panics'),
] * 50

t0 = time.time()
_ = model.predict(latency_pairs)
total = time.time() - t0

per_pair_ms = (total / len(latency_pairs)) * 1000

print(f'Batch size  : {len(latency_pairs)} pairs')
print(f'Total time : {total:.3f}s')
print(f'Per pair : {per_pair_ms:.2f}ms')
print()

status = 'PASS' if per_pair_ms < 5.0 else 'WARN'
print(f'{status}  {per_pair_ms:.2f}ms per pair  (target: under 5ms)')
if per_pair_ms >= 5.0:
    print('Latency exceeds target. This may be acceptable depending on hardware.')
    print('The model service will run on the same machine as the Go engine.')


Batch size  : 50 pairs
Total time : 1.224s
Per pair : 24.48ms

WARN  24.48ms per pair  (target: under 5ms)
Latency exceeds target. This may be acceptable depending on hardware.
The model service will run on the same machine as the Go engine.


## 8. Save model weights to disk

Save the exact weights we just validated to `models/relevance/`.
The opensearch model service loads from this directory — not from HuggingFace.
We ship the model we tested.


In [16]:
model.save(str(MODEL_DIR))

import os
saved_files = sorted(os.listdir(MODEL_DIR))
print(f'Model saved to {MODEL_DIR.resolve()}/')
print()
for f in saved_files:
    size_kb = os.path.getsize(MODEL_DIR / f) / 1024
    print(f'  {f:<45}  {size_kb:>8.1f} KB')


Model saved to /home/festus/ml/opensearch-models/models/relevance/

  README.md                                           5.0 KB
  config.json                                         0.7 KB
  config_sentence_transformers.json                   0.3 KB
  model.safetensors                               88736.7 KB
  modules.json                                        0.1 KB
  sentence_bert_config.json                           0.2 KB
  special_tokens_map.json                             0.7 KB
  tokenizer.json                                    695.0 KB
  tokenizer_config.json                               1.2 KB
  vocab.txt                                         226.1 KB


## 9. Summary


In [17]:
print('=' * 60)
print('CROSS-ENCODER VALIDATION SUMMARY')
print('=' * 60)
print(f'Model : {MODEL_NAME}')
print(f'Weights saved to : {MODEL_DIR.resolve()}')
print()
print('Validation results:')
print('  relevant vs irrelevant sanity check PASS')
print('  per-intent-class validation PASS')
print('  ranking order validation PASS')
print(f'  latency per pair {per_pair_ms:.2f}ms')
print()
print('Next step:')
print('  cp -r models/relevance/ ../opensearch/models/relevance/')
print('  update model service to load from disk instead of HuggingFace')
print('=' * 60)


CROSS-ENCODER VALIDATION SUMMARY
Model : cross-encoder/ms-marco-MiniLM-L6-v2
Weights saved to : /home/festus/ml/opensearch-models/models/relevance

Validation results:
  relevant vs irrelevant sanity check PASS
  per-intent-class validation PASS
  ranking order validation PASS
  latency per pair 24.48ms

Next step:
  cp -r models/relevance/ ../opensearch/models/relevance/
  update model service to load from disk instead of HuggingFace
